In [ ]:

import pandas as pd
import pathlib
import glob
import os
import numpy as np
import warnings
import gc  # Added for memory management
import pyanalib.split_df_helpers as splh
from tables import NaturalNameWarning

# 1. Suppress warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=NaturalNameWarning)
from tqdm import tqdm # Standard for progress bars in SBND environments

In [ ]:
'''
def consolidate_buffered(folder_path, output_name, split_margin_gb=1.0, num_splits=1, shift_ntuple_index=False):
    """
    Consolidates files with robust global ntuple index shifting implemented via vector tracking.
    """
    out_dir = pathlib.Path("/exp/sbnd/data/users/lpelegri/cafpyana_data")
    out_dir.mkdir(parents=True, exist_ok=True)
    output_file = out_dir / f"{output_name}.df"

    input_folder = pathlib.Path(folder_path)
    files = sorted(glob.glob(str(input_folder / f"{output_name}_*.df")))
    
    if not files:
        print(f"No files found for {output_name}")
        return

    # --- ANNOUNCE START ---
    print(f"\n{'='*60}")
    print(f"COMBINING: {output_name}")
    print(f"TARGET:    {output_file}")
    print(f"SHIFT INDEX: {shift_ntuple_index}")
    print(f"TOTAL FILES TO PROCESS: {len(files)}")
    print(f"{'='*60}\n")

    # 1. Discover keys
    with pd.HDFStore(files[0], mode='r') as store:
        keys2load = [k.lstrip('/').rsplit('_', 1)[0] for k in store.keys() if 'split' not in k]
        keys2load = sorted(list(set(keys2load)))

    df_buffers = {k: [] for k in keys2load}
    size_counters = {k: 0 for k in keys2load}
    k_idx = 0 
    total_ntuples_seen = 0 

    with pd.HDFStore(output_file, mode='w') as hdf_out:
        pbar = tqdm(files, desc="Consolidating", unit="file")
        
        for file_path in pbar:
            if k_idx >= num_splits:
                pbar.write(f"Reached limit of {num_splits} splits. Stopping.")
                break
            
            file_data = splh.load_dfs(file_path, keys2load)
            
            # --- Robust Index Shifting Logic (Ported from Big Files function) ---
            if shift_ntuple_index:
                # Find the maximum absolute __ntuple index value across ALL dataframes in this file block
                max_ntuple_in_file = -1
                for k, df in file_data.items():
                    if df is not None and not df.empty:
                        max_val = df.index.get_level_values("__ntuple").max()
                        if max_val > max_ntuple_in_file:
                            max_ntuple_in_file = max_val
                
                # The offset distance is the max index value plus 1
                num_in_this_file = int(max_ntuple_in_file + 1) if max_ntuple_in_file >= 0 else 0

                if total_ntuples_seen > 0:
                    for k, df in file_data.items():
                        if df is not None and not df.empty:
                            # Safe multi-index manipulation via vectorized tracking dataframes
                            idx_df = df.index.to_frame()
                            idx_df["__ntuple"] += total_ntuples_seen
                            
                            # Re-assign the safely shifted MultiIndex back to the DataFrame
                            df.index = pd.MultiIndex.from_frame(idx_df)
                
                # Increment the running counter by the true safety boundary limit
                total_ntuples_seen += num_in_this_file

            # --- BUFFERING ---
            for k in keys2load:
                df = file_data[k]
                if df is not None and not df.empty:
                    size_gb = df.memory_usage(deep=True).sum() / (1024**3)
                    size_counters[k] += size_gb
                    df_buffers[k].append(df)
            
            # --- FLUSH LOGIC ---
            current_total_buffer_gb = sum(size_counters.values())
            if current_total_buffer_gb >= split_margin_gb:
                pbar.write(f"--- Buffer limit reached ({current_total_buffer_gb:.3f} GB). Flushing to split_{k_idx} ---")
                for k in keys2load:
                    if df_buffers[k]:
                        concat_df = pd.concat(df_buffers[k], ignore_index=False)
                        hdf_out.put(key=f"{k}_{k_idx}", value=concat_df, format="fixed")
                        df_buffers[k] = [] 
                        size_counters[k] = 0
                k_idx += 1
                gc.collect()

                if k_idx >= num_splits:
                    break
                
            del file_data
            gc.collect()

        # Final Flush for remaining data
        if k_idx < num_splits and any(len(b) > 0 for b in df_buffers.values()):
            print(f"--- Final Flush: split_{k_idx} ---")
            for k in keys2load:
                if df_buffers[k]:
                    concat_df = pd.concat(df_buffers[k], ignore_index=False)
                    hdf_out.put(key=f"{k}_{k_idx}", value=concat_df, format="fixed")
            k_idx += 1
            gc.collect()

        hdf_out.put(key="split", value=pd.DataFrame({"n_split": [k_idx]}), format="fixed")

    print(f"\nConsolidation finished. Total splits: {k_idx}")
    
    # Cleanup block
    print("Deleting temporary files...")
    for file_path in files:
        try:
            os.remove(file_path)
        except OSError:
            pass 
    print("All Clean!")
    return output_file
    
'''

In [ ]:
import pandas as pd
import pathlib
import glob
import os
import gc
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, wait, FIRST_COMPLETED
import pyanalib.split_df_helpers as splh
from tqdm import tqdm


def _load_single_file(file_path, keys2load):
    """Worker: ONLY loads the dataframes for one file."""
    import numpy.core.multiarray  # Prevents late-import decorator collision
    try:
        data = splh.load_dfs(file_path, keys2load)
        return file_path, data
    except Exception as e:
        import traceback
        return file_path, f"{str(e)}\n{traceback.format_exc()}"


def consolidate_buffered(folder_path, output_name, output_folder = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData", split_margin_gb=1.0, num_splits=1,
                          shift_ntuple_index=False, max_workers=None, max_in_flight=None):
    """
    Consolidates files with robust global ntuple index shifting implemented via vector tracking.

    Loading is parallelized with a BOUNDED rolling window: at most `max_in_flight`
    files are ever submitted to the worker pool at once (a new one goes in only
    as an old one finishes), instead of submitting everything up front. This keeps
    the backlog of completed-but-undrained results small, which makes Ctrl+C/stop
    responsive and avoids the "wall of progress messages" a huge simultaneous
    backlog causes. Writes are still folded in strictly in original file order
    (via next_file_to_write), so __ntuple offsets and split boundaries are exactly
    as correct as the single-threaded version - the rolling window changes nothing
    about ordering, only how much work is ever queued up at once.
    """
    total_cpus = multiprocessing.cpu_count()
    use_cpus = max_workers if max_workers is not None else max(1, int(total_cpus * 0.7))
    max_in_flight = max_in_flight if max_in_flight is not None else use_cpus * 2

    print(f"\n{'='*60}")
    print(f"SYSTEM: {total_cpus} CPUs. Using {use_cpus} workers ({max_in_flight} files in flight at a time).")
    print(f"{'='*60}\n")

    out_dir = pathlib.Path(output_folder)
    out_dir.mkdir(parents=True, exist_ok=True)
    output_file = out_dir / f"{output_name}.df"

    if output_file.exists():
        output_file.unlink()

    input_folder = pathlib.Path(folder_path)
    files = sorted(glob.glob(str(input_folder / f"{output_name}_*.df")))

    if not files:
        print(f"No files found for {output_name}")
        return

    print(f"\n{'='*60}")
    print(f"COMBINING: {output_name}")
    print(f"TARGET:    {output_file}")
    print(f"SHIFT INDEX: {shift_ntuple_index}")
    print(f"TOTAL FILES TO PROCESS: {len(files)}")
    print(f"{'='*60}\n")

    # 1. Discover keys
    with pd.HDFStore(files[0], mode='r') as store:
        keys2load = sorted(list(set([k.lstrip('/').rsplit('_', 1)[0] for k in store.keys() if 'split' not in k])))

    df_buffers = {k: [] for k in keys2load}
    size_counters = {k: 0 for k in keys2load}
    k_idx, total_ntuples_seen = 0, 0
    reached_limit = False
    consolidated_files = []  # only files actually folded into the output get deleted later

    n_files = len(files)
    pending = {}          # future -> file index
    results_ready = {}     # file index -> (file_path, file_data) | None
    next_file_to_write = 0
    next_idx_to_submit = 0

    executor = ProcessPoolExecutor(max_workers=use_cpus)

    def _submit_next():
        nonlocal next_idx_to_submit
        if next_idx_to_submit < n_files:
            fut = executor.submit(_load_single_file, files[next_idx_to_submit], keys2load)
            pending[fut] = next_idx_to_submit
            next_idx_to_submit += 1

    pbar = tqdm(total=n_files, desc="Consolidating", unit="file", mininterval=1.0)
    try:
        # seed the initial rolling window
        for _ in range(min(max_in_flight, n_files)):
            _submit_next()

        while pending and not reached_limit:
            done, _ = wait(pending.keys(), return_when=FIRST_COMPLETED)

            for future in done:
                idx = pending.pop(future)  # release the reference immediately
                file_path, result = future.result()

                if isinstance(result, str):
                    pbar.write(f"Error in {file_path}: {result}")
                    results_ready[idx] = None
                else:
                    results_ready[idx] = (file_path, result)

                # keep the window full - only submit as one finishes
                _submit_next()

            # Drain strictly in original order - this is what keeps
            # index shifting / split boundaries correct.
            while next_file_to_write in results_ready:
                entry = results_ready.pop(next_file_to_write)
                next_file_to_write += 1
                pbar.update(1)

                if entry is None:  # failed file, skip
                    continue

                file_path, file_data = entry
                consolidated_files.append(file_path)

                # --- Robust Index Shifting Logic ---
                if shift_ntuple_index:
                    max_ntuple_in_file = -1
                    for k, df in file_data.items():
                        if df is not None and not df.empty:
                            max_val = df.index.get_level_values("__ntuple").max()
                            if max_val > max_ntuple_in_file:
                                max_ntuple_in_file = max_val

                    num_in_this_file = int(max_ntuple_in_file + 1) if max_ntuple_in_file >= 0 else 0

                    if total_ntuples_seen > 0:
                        for k, df in file_data.items():
                            if df is not None and not df.empty:
                                idx_df = df.index.to_frame()
                                idx_df["__ntuple"] += total_ntuples_seen
                                df.index = pd.MultiIndex.from_frame(idx_df)

                    total_ntuples_seen += num_in_this_file

                # --- BUFFERING ---
                for k in keys2load:
                    if k in file_data and file_data[k] is not None and not file_data[k].empty:
                        size_counters[k] += file_data[k].values.nbytes / (1024**3)
                        df_buffers[k].append(file_data[k])

                current_buffer_gb = sum(size_counters.values())

                # --- FLUSH LOGIC (only place we touch postfix / print progress) ---
                if current_buffer_gb >= split_margin_gb:
                    pbar.set_postfix({"buf_GB": f"{current_buffer_gb:.2f}", "split": k_idx})
                    pbar.write(f"--- Flushing split_{k_idx} ---")
                    with pd.HDFStore(output_file, mode='a') as hdf_out:
                        for k in keys2load:
                            if df_buffers[k]:
                                concat_df = pd.concat(df_buffers[k], ignore_index=False)
                                hdf_out.put(key=f"{k}_{k_idx}", value=concat_df, format="fixed")
                                df_buffers[k] = []
                                size_counters[k] = 0
                                del concat_df
                    k_idx += 1
                    gc.collect()

                    if num_splits and k_idx >= num_splits:
                        reached_limit = True
                        break  # exits while loop

                del file_data

            if reached_limit:
                break  # exits outer while loop

    except KeyboardInterrupt:
        pbar.write(f"\nInterrupted. {k_idx} split(s) already flushed to disk are complete and "
                    f"correctly ordered. Any not-yet-flushed buffered data was discarded.")
        raise
    finally:
        pbar.close()
        executor.shutdown(wait=False, cancel_futures=True)
        gc.collect()

    # Final Flush for remaining data (only reached on a clean, non-interrupted finish)
    if not reached_limit and any(len(b) > 0 for b in df_buffers.values()):
        print(f"--- Final Flush: split_{k_idx} ---")
        with pd.HDFStore(output_file, mode='a') as hdf_out:
            for k in keys2load:
                if df_buffers[k]:
                    hdf_out.put(key=f"{k}_{k_idx}", value=pd.concat(df_buffers[k]), format="fixed")
        k_idx += 1

    # Metadata
    with pd.HDFStore(output_file, mode='a') as hdf_out:
        hdf_out.put(key="split", value=pd.DataFrame({"n_split": [k_idx]}), format="fixed")

    print(f"\nConsolidation finished. Total splits: {k_idx}")

    # Cleanup block - only remove files that actually made it into the output
    print("Deleting temporary files...")
    for file_path in consolidated_files:
        try:
            os.remove(file_path)
        except OSError:
            pass
    print("All Clean!")
    return output_file

In [ ]:
# --- Execution ---
# Dictionary mapping file names to their shift_ntuple_index status
'''
file_config = {
    "cc1pi_5e18_CV": False,
    "mc_GIBUU_gen1": False,
    "cc1pi_data_rollingdev_bnblight": False, # Usually False for data
    "cc1pi_data_offbeamlight": False,        # Usually False for data
    "cc1pi_1e20_lowE_CV": True,
    "cc1pi_5e18_in_time_cosmics": False,
    "cc1pi_5e18_CV_extra_syst": False,
    "cc1pi_extended_syst": False,
    "cc1pi_SystVarsCV": False,
    "cc1pi_ccalp": False,
    "cc1pi_alphap": False,
    "cc1pi_ccalm": False,
    "cc1pi_alpham": False,
    "cc1pi_betam": False,
    "cc1pi_rm": False,
    "cc1pi_rp": False,
    "cc1pi_betap": False,
    "cc1pi_wiremod_YZ": False,
    "cc1pi_wiremod_XZ_thetaXW": False,
    "cc1pi_0xSCE": False,
    "cc1pi_2xSCE": False,
    "cc1pi_PMTHighNoise": False,
    "cc1pi_PMTGainFluct": False,
    "cc1pi_PMTLowEff": False
}
'''

'''
output_folder = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData"
file_config = {
    "TLE_1e20_pion_stopping_update_calo": False, # Usually False for data
    "TLE_1e20_pion_data_update_calo": False, # Usually False for data
    #"TLE_1e20_pion_data": False, # Usually False for data
    #"TLE_5e18_muon_data": False, # Usually False for data
    #"TLE_1e20_muon": False, # Usually False for data
    #"TLE_1e20_pion_all": False, # Usually False for data
    #"TLE_1e20_pion_stopping": False, # Usually False for data
}
'''

output_folder = "/exp/sbnd/data/users/lpelegri/cafpyana_data"
file_config = {
    "cc1pi_1e20_training_update_cv": False,
    "cc1pi_data_fixdev_bnblight_update_cv": False,
}

SOURCE = "/exp/sbnd/data/users/lpelegri/cafpyana_data_transfer_folder"

for file_name, should_shift in file_config.items():
    print(f"\n>>> Processing {file_name} (Shift Index: {should_shift})")
    
    consolidate_buffered(
        SOURCE, 
        file_name, 
        output_folder = output_folder,
        split_margin_gb=1, 
        num_splits=1000, 
        shift_ntuple_index=True
    )

In [ ]:
import hashlib
from makedf.mcstat import get_MCstat_unc
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
import hashlib
def get_MCstat_unc(evt_df, hdr_df, n_universes=100):
    # Create a unique seed based on event metadata
    # Using a hash function that's deterministic
    meta_seeds = []
    for i in range(len(evt_df)):        
        this_hdr_df = hdr_df.loc[evt_df.reset_index(level=[2]).index[i]]
        runno = this_hdr_df.run
        subrunno = this_hdr_df.subrun
        evtno = this_hdr_df.evt
        slcid = evt_df.loc[evt_df.index[i]].slc.self
        seed_string = f"run_{runno}_subrun_{subrunno}_evt_{evtno}_slcid_{slcid}"
        #unique_seed = hash(f"run_{runno}_subrun_{subrunno}_evt_{evtno}_slcid_{slcid}") % (2**32)  # Ensure it's a 32-bit integer
        unique_seed = int(
            hashlib.sha256(seed_string.encode()).hexdigest(),
            16
        ) % (2**32)
        if unique_seed in meta_seeds:
            print("duplicate seed found", unique_seed)
            break
        meta_seeds.append(unique_seed)

    # make sure the seeds are unique!
    assert len(meta_seeds) == len(set(meta_seeds))

    # generate universes
    MCstat_univ_events = np.zeros((n_universes, len(evt_df)))
    poisson_mean = 1.0

    # get Poisson weights and save to "MCstat.univ_"
    # dummy df to hold the weights -- iterative inserting causes PerformanceWarning
    mcstat_univ_cols = pd.MultiIndex.from_product(
        [["truth"], ["MCstat"], [f"univ_{i}" for i in range(n_universes)],[""],[""],[""]],
    )
    mcstat_univ_wgt = pd.DataFrame(
        1.0,
        index=evt_df.index,
        columns=mcstat_univ_cols,
    )

    for uidx in range(n_universes):
        universe_string = f"universe_{uidx}"
        universe_seed = int(
            hashlib.sha256(universe_string.encode()).hexdigest(),
            16
        ) % (2**32)
            
        poisson_weights = []
        for sidx, meta_seed in enumerate(meta_seeds):
            # Combine universe seed with event seed for unique randomness -- per event, per universe
            combined_seed = (universe_seed + meta_seed) % (2**32)
            np.random.seed(combined_seed)
            
            poisson_val = np.random.poisson(poisson_mean)
            poisson_weights.append(poisson_val)
            
        mcstat_univ_wgt[("truth","MCstat", "univ_{}".format(uidx),'','','')] = np.array(poisson_weights)
        MCstat_univ_events[uidx, :] = np.array(poisson_weights)

    evt_df = evt_df.join(mcstat_univ_wgt)
    return evt_df, MCstat_univ_events

In [ ]:
import pandas as pd
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.CutMasks import CutMasks
from analysis_village.cc1pi.Constants import CTE

import numpy as np


def TruthInFV(data):
    x_region = (np.abs(data.x) > 5) & (np.abs(data.x) < 190)
    z_region1 = (data.z > 10)  & (data.z < 250) & (np.abs(data.y) < 190)
    z_region2 = (data.z > 250) & (data.z < 450) & (data.y > -190) & (data.y < 100) & (data.x < 0)
    z_region3 = (data.z > 250) & (data.z < 450) & (data.y > -190) & (data.y < 190) & (data.x > 0)
    
    contained = x_region & (z_region1 | z_region2 | z_region3)
    return contained

    
def IsNu(df):
    is_numu = abs(df.pdg) == 14
    is_nue = abs(df.pdg) == 12
    return is_numu | is_nue   

def isCC1Pi(df): # definition
    is_1pi1mu = (df.nmu_P_100MeV_3000MeV == 1) & (df.npi_P_130MeV_2000MeV == 1) & (df.npi_P_85MeV_10000MeV == 1)
    is_NpiNmuNnNp = df.nprim - df.nmu - df.npi - df.np - df.nn == 0

    # Initialize full theta mask (False by default)
    is_theta = pd.Series(False, index=df.index)

    # Only compute angles where needed
    df_sel = df.loc[is_1pi1mu]
    
    if len(df_sel) > 0:
        cpi_vec = df_sel.loc[:, ('cpi','genp',['x','y','z'])].to_numpy()
        mu_vec  = df_sel.loc[:, ('mu','genp',['x','y','z'])].to_numpy()
     

        mu_mag  = np.linalg.norm(mu_vec, axis=1)
        cpi_mag = np.linalg.norm(cpi_vec, axis=1)
        dot     = np.sum(mu_vec * cpi_vec, axis=1)

        cos_theta = dot / np.clip(mu_mag * cpi_mag, 1e-12, None)
        theta     = np.arccos(np.clip(cos_theta, -1.0, 1.0))
            
        # Assign back using the SAME index subset
        is_theta.loc[df_sel.index] = theta < CTE.max_angle_between_candidates

    is_mu_p =  df.mu.totp < 1
    #return is_1pi1mu & is_NpiNmuNnNp & is_theta & is_mu_contained
    return is_1pi1mu & is_NpiNmuNnNp & is_theta & is_mu_p

def add_nu_categ_column(df, is_truth_df = False):
    if(is_truth_df):
        truth_df = df
    else:
        truth_df = df.slc.truth # Make a copy to safely assign

    is_inside_fv = TruthInFV(truth_df.position)
    is_nu = IsNu(truth_df)
    is_signal = isCC1Pi(truth_df)
    is_cc = truth_df.iscc
    is_nu_mu_cc = is_cc & (abs(truth_df.pdg) == 14)

    nu_categ = pd.Series("none", index=truth_df.index, dtype="object")
    # Apply categories
    nu_categ[~is_nu] = "cosmic"
    nu_categ[is_nu & ~is_inside_fv] = "out_AV_nu"
    nu_categ[is_nu & is_inside_fv & ~is_cc] = "NC"
    nu_categ[is_nu & is_inside_fv & is_cc & (abs(truth_df.pdg) == 12)] = "CC_e"
    nu_categ[is_nu & is_inside_fv & is_nu_mu_cc & (truth_df.npi_P_85MeV_10000MeV == 0) & (truth_df.np_P_325MeV_10000MeV == 1)] = "CC_mu_0pi_1p"
    nu_categ[is_nu & is_inside_fv & is_nu_mu_cc & (truth_df.npi_P_85MeV_10000MeV == 0) & (truth_df.np_P_325MeV_10000MeV > 1)] = "CC_mu_0pi_2p"
    nu_categ[is_nu & is_inside_fv & is_nu_mu_cc & (truth_df.npi_P_85MeV_10000MeV == 0) & (truth_df.np_P_325MeV_10000MeV == 0)] = "CC_mu_0pi_0p"
    nu_categ[is_nu & is_inside_fv & is_nu_mu_cc & (truth_df.npi_P_85MeV_10000MeV > 1)] = "CC_mu_2pi"
    nu_categ[is_nu & is_inside_fv & is_nu_mu_cc & is_signal] = "CC1pi"
    nu_categ[is_nu & is_inside_fv & is_nu_mu_cc & ~is_signal & (truth_df.npi_P_85MeV_10000MeV == 1)] = "other_CC1pi"
    
    
    if(is_truth_df):
        df['nu_categ'] = nu_categ
    else:
        df[('slc','truth', 'nu_categ', '', '','')] = nu_categ 
    return df

   
def add_nu_categ_proton_reduced_column(df, is_truth_df = False):
    if(is_truth_df):
        truth_df = df
    else:
        truth_df = df.truth # Make a copy to safely assign

    is_inside_fv = TruthInFV(truth_df.position)
    is_nu = IsNu(truth_df)
    is_signal = isCC1Pi(truth_df)
    is_cc = truth_df.iscc.astype(bool)
    is_nu_mu_cc = is_cc & (abs(truth_df.pdg) == 14)

    nu_categ = pd.Series("none", index=truth_df.index, dtype="object")
    # Apply categories
    nu_categ[~is_nu] = "cosmic"
    nu_categ[is_nu & ~is_inside_fv] = "out_AV_nu"
    nu_categ[is_nu & is_inside_fv & ~is_signal] = "other_nu"
    nu_categ[is_nu & is_inside_fv & is_nu_mu_cc & (truth_df.npi_P_85MeV_10000MeV == 0)] = "CC_mu_0pi"
    nu_categ[is_nu & is_inside_fv & is_nu_mu_cc & (truth_df.npi_P_85MeV_10000MeV > 1)] = "CC_mu_2pi"
    nu_categ[is_nu & is_inside_fv & is_nu_mu_cc & is_signal & (truth_df.np_P_325MeV_10000MeV == 0)] = "0p_CC1Pi"
    nu_categ[is_nu & is_inside_fv & is_nu_mu_cc & is_signal & (truth_df.np_P_325MeV_10000MeV == 1)] = "1p_CC1Pi"
    nu_categ[is_nu & is_inside_fv & is_nu_mu_cc & is_signal & (truth_df.np_P_325MeV_10000MeV > 1)] = "plus2p_CC1Pi"
    
    if(is_truth_df):
        df['nu_categ_proton_reduced'] = nu_categ
    else:
        df[('truth', 'nu_categ_proton_reduced', '', '','','')] = nu_categ 
    return df

def add_genie_categ_column(df, is_truth_df = False):
    if(is_truth_df):
        truth_df = df
    else:
        truth_df = df.truth # Make a copy to safely assign

    is_inside_fv = TruthInFV(truth_df.position)
    is_nu = IsNu(truth_df)
    is_cc = truth_df.iscc.astype(bool)
    is_nu_mu_cc = is_cc & (abs(truth_df.pdg) == 14)

    genie_categ = pd.Series("other", index=truth_df.index, dtype="object")
    # Apply categories
    genie_categ[~is_nu] = "cosmic"
    genie_categ[is_nu & ~is_inside_fv] = "out_AV_nu"
    genie_categ[is_nu & is_inside_fv & ~is_cc & (abs(truth_df.pdg) == 14)] = "nu_mu_NC" 
    genie_categ[is_nu & is_inside_fv & is_nu_mu_cc & (df.genie_mode == 0)] = "nu_mu_CC_QE" 
    genie_categ[is_nu & is_inside_fv & is_nu_mu_cc & (df.genie_mode == 10)] = "nu_mu_CC_MEC" 
    genie_categ[is_nu & is_inside_fv & is_nu_mu_cc & (df.genie_mode == 1)] = "nu_mu_CC_Res" 
    genie_categ[is_nu & is_inside_fv & is_nu_mu_cc & (df.genie_mode == 2)] = "nu_mu_CC_Dis" 
    
    if(is_truth_df):
        df['genie_categ'] = genie_categ
    else:
        df[('truth', 'genie_categ', '', '','','')] = nu_categ 
    return df

In [ ]:
import pandas as pd
import pathlib
import glob
import os
import numpy as np
import warnings
import gc
import pyanalib.split_df_helpers as splh
from tables import NaturalNameWarning
from tqdm import tqdm

# Selection/Mask imports
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.CutMasks import CutMasks

# Suppress common HDF5/Pandas warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=NaturalNameWarning)

import os
# Force HDF5 to ignore file locks - essential for cluster/multi-process work
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'


import pandas as pd
import pathlib
import glob
import os
import numpy as np
import warnings
import gc
import pyanalib.split_df_helpers as splh
from tables import NaturalNameWarning
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing

# Suppress warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=NaturalNameWarning)

from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.CutMasks import CutMasks

def process_single_file(file_path, keys2load, filter_df):
    try:
        data = splh.load_dfs(file_path, keys2load, 10000000)
        data['cc1pi'][('slc', 'cut', 'proton_BDT_2pi', '', '', '')] = CutMasks.proton_BDT_cut_mask_2pi(data['cc1pi'], ['__ntuple', 'entry', 'rec.slc..index'])
        data['cc1pi'][('slc', 'cut', 'proton_BDT_sideband', '', '', '')] = CutMasks.proton_BDT_sideband_mask(data['cc1pi'], ['__ntuple', 'entry', 'rec.slc..index'])
        data['cc1pi'][('slc', 'cut', 'TPC_containment', '', '', '')] = CutMasks.TPC_containment_mask(data['cc1pi'], ['__ntuple', 'entry', 'rec.slc..index'])
        data['cc1pi'][('slc', 'cut', 'inside_FV', '', '', '')] = CutMasks.InFV_strict(data['cc1pi'])
        data['cc1pi'][('slc', 'cut', 'no_high_yz', '', '', '')] = CutMasks.not_in_high_y_high_z_containment_mask(data['cc1pi'], ['__ntuple', 'entry', 'rec.slc..index'])  
        data['cc1pi'][('slc', 'cut', 'energy', '', '', '')] = (data['cc1pi'].slc.measure_var.reco_p_mu > 0.1) & (data['cc1pi'].slc.measure_var.reco_p_mu < 1) & (data['cc1pi'].slc.measure_var.TLE_p_pi > 0.13) & (data['cc1pi'].slc.measure_var.TLE_p_pi < 2)   
        data['nudf'] = data['nudf'].loc[~data['nudf'].index.duplicated(keep='first')]
           
        data['nudf'] = add_nu_categ_column(data['nudf'], True)
        data['nudf'] = add_nu_categ_proton_reduced_column(data['nudf'], True)
        data['nudf'] = add_genie_categ_column(data['nudf'], True)
 
        
        if filter_df and "cc1pi" in data:
            cc1pi = data["cc1pi"]
            if cc1pi is not None and not cc1pi.empty:
                # 2. Build Masks & Filter
                m_signal = build_event_cumulative_masks(cc1pi, sideband="")["energy"]
                m_2pi = build_event_cumulative_masks(cc1pi, sideband="two_pions")["energy"]
                m_prot = build_event_cumulative_masks(cc1pi, sideband="proton")["energy"]
                cc1pi = cc1pi[m_signal | m_2pi | m_prot]
                
                # 3. Clean up Indexing
                cc1pi = cc1pi.groupby(['__ntuple', 'entry', 'rec.slc..index']).first().sort_index()
                data["cc1pi"] = cc1pi
                data["cc1pi"],_ = get_MCstat_unc(data["cc1pi"], data["hdr"], n_universes=100)
                
                # 4. TRUTH FILTERING (Revised for strict matching)
                if "nudf" in data and data["nudf"] is not None:
                    nudf = data["nudf"]
                    tmatch_col = ('slc', 'tmatch', 'idx', '', '', '')
                    
                    # Create the valid set of (ntuple, entry, nu_index)
                    # We drop NaNs or -1 values to avoid "fake" matches
                    valid_reco = cc1pi[cc1pi[tmatch_col] >= 0]
                    valid_set = set(zip(
                        valid_reco.index.get_level_values('__ntuple'),
                        valid_reco.index.get_level_values('entry'),
                        valid_reco[tmatch_col].astype(int)
                    ))

                    nu_categ_col = ('nu_categ', '', '')
                    # Rule A: Keep it if it's a Signal category (for efficiency denominator)
                    mask_signal_truth = (nudf[nu_categ_col] == "CC1pi")
                    
                    # Rule B: Keep it if it's the truth for ANY slice we kept in cc1pi
                    # This prevents the "orphan slice" error
                    nudf_indices = zip(
                        nudf.index.get_level_values('__ntuple'),
                        nudf.index.get_level_values('entry'),
                        nudf.index.get_level_values('rec.mc.nu..index')
                    )
                    mask_is_matched = np.array([tup in valid_set for tup in nudf_indices])
                    
                    data["nudf"] = nudf[mask_signal_truth | mask_is_matched]
        return file_path, data
    except Exception as e:
        import traceback
        return file_path, f"{str(e)}\n{traceback.format_exc()}"


import pandas as pd
import pathlib
import glob
import gc
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

def consolidate_buffered_big_files(folder_path, output_name, split_margin_gb=1.0, num_splits=None, num_files=None, shift_ntuple_index=True, filter_df=True):
    # --- CPU & RAM SAFETY ---
    total_cpus = multiprocessing.cpu_count()
    use_cpus = max(1, int(total_cpus * 0.7))
    print(f"\n{'='*60}")
    print(f"SYSTEM: {total_cpus} CPUs. Using {use_cpus} workers.")
    print(f"OUTPUT: Single file at {output_name}.df")
    print(f"{'='*60}\n")

    out_dir = pathlib.Path("/exp/sbnd/data/users/lpelegri/cafpyana_data")
    out_dir.mkdir(parents=True, exist_ok=True)
    output_file = out_dir / f"{output_name}_w_proton.df"

    if output_file.exists():
        output_file.unlink()

    input_folder = pathlib.Path(folder_path)
    search_pattern = str(input_folder / f"{output_name}_[0-9]*.df")
    files = sorted(glob.glob(search_pattern))

    if num_files is not None:
        files = files[:num_files]

    if not files:
        print("No files found.")
        return

    with pd.HDFStore(files[0], mode='r') as store:
        keys2load = sorted(list(set([k.lstrip('/').rsplit('_', 1)[0] for k in store.keys() if 'split' not in k])))

    df_buffers = {k: [] for k in keys2load}
    size_counters = {k: 0 for k in keys2load}
    k_idx, total_ntuples_seen = 0, 0
    reached_limit = False

    executor = ProcessPoolExecutor(max_workers=use_cpus)
    try:
        future_to_file = {executor.submit(process_single_file, f, keys2load, filter_df): i for i, f in enumerate(files)}
        results_ready = {}
        next_file_to_write = 0
        pbar = tqdm(total=len(files), desc="Consolidating", unit="file")

        for future in as_completed(future_to_file):
            file_path, result = future.result()
            if isinstance(result, str):
                pbar.write(f"Error in {file_path}: {result}")
                results_ready[future_to_file[future]] = None  # sentinel
                continue

            results_ready[future_to_file[future]] = result

            # Sequential processing to maintain index integrity
            while next_file_to_write in results_ready:
                file_data = results_ready.pop(next_file_to_write)
                next_file_to_write += 1
                pbar.update(1)

                if file_data is None:  # failed file, skip
                    continue

                if shift_ntuple_index:
                    # Find the maximum absolute __ntuple index value across ALL dataframes in this file block
                    max_ntuple_in_file = -1
                    for k, df in file_data.items():
                        if df is not None and not df.empty:
                            max_val = df.index.get_level_values("__ntuple").max()
                            if max_val > max_ntuple_in_file:
                                max_ntuple_in_file = max_val
                    
                    # The offset distance is the max index value plus 1
                    num_in_this_file = int(max_ntuple_in_file + 1) if max_ntuple_in_file >= 0 else 0

                    if total_ntuples_seen > 0:
                        for k, df in file_data.items():
                            if df is not None and not df.empty:
                                # Safe multi-index manipulation via tracking frames
                                idx_df = df.index.to_frame()
                                idx_df["__ntuple"] += total_ntuples_seen
                                
                                # Re-assign the safely shifted MultiIndex back to the DataFrame
                                df.index = pd.MultiIndex.from_frame(idx_df)
                                
                    # Increment the counter by the true safety boundary limit
                    total_ntuples_seen += num_in_this_file

                for k in keys2load:
                    if k in file_data and file_data[k] is not None and not file_data[k].empty:
                        size_counters[k] += file_data[k].values.nbytes / (1024**3)
                        df_buffers[k].append(file_data[k])

                current_buffer_gb = sum(size_counters.values())
                pbar.set_postfix({"buf_GB": f"{current_buffer_gb:.2f}", "split": k_idx})

                if current_buffer_gb >= split_margin_gb:
                    pbar.write(f"--- Flushing split_{k_idx} ---")
                    with pd.HDFStore(output_file, mode='a') as hdf_out:
                        for k in keys2load:
                            if df_buffers[k]:
                                concat_df = pd.concat(df_buffers[k], ignore_index=False)
                                hdf_out.put(key=f"{k}_{k_idx}", value=concat_df, format="fixed")
                                df_buffers[k] = []
                                size_counters[k] = 0
                                del concat_df
                    k_idx += 1
                    gc.collect()

                    if num_splits and k_idx >= num_splits:
                        reached_limit = True
                        break  # exits while loop

            if reached_limit:
                break  # exits for loop

    finally:
        executor.shutdown(wait=False, cancel_futures=True)
        gc.collect()

    # Final Flush for remaining data
    if not reached_limit and any(len(b) > 0 for b in df_buffers.values()):
        pbar.write(f"--- Final Flush: split_{k_idx} ---")
        with pd.HDFStore(output_file, mode='a') as hdf_out:
            for k in keys2load:
                if df_buffers[k]:
                    hdf_out.put(key=f"{k}_{k_idx}", value=pd.concat(df_buffers[k]), format="fixed")
        k_idx += 1

    # Metadata
    with pd.HDFStore(output_file, mode='a') as hdf_out:
        hdf_out.put(key="split", value=pd.DataFrame({"n_split": [k_idx]}), format="fixed")

    print(f"\nDone! Single consolidated file created with {k_idx} splits.")
    return output_file

In [ ]:

file_config = {
    #"mc_ar23p": False, # Usually False for data
    "mc_ar23p_extended_syst": True, # Usually False for data
}


SOURCE = "/exp/sbnd/data/users/lpelegri/cafpyana_data_transfer_folder"

for file_name, filter_df in file_config.items():
    consolidate_buffered_big_files(SOURCE, file_name, split_margin_gb=1, num_splits=5, shift_ntuple_index = True, num_files = None, filter_df=filter_df)

In [ ]:
from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from cols_to_keep import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.CutMasks.MaskUtils import *
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file

#_for_data_mc_comp, _no_syst, _min_reco_no_syst
key = "_min_reco_no_syst"

n_split = 1000
#df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/mc_ar23p_extended_syst.df", keys2load, n_split, reprocess_df = False, reprocess_truth = True)
df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data_transfer_folder/mc_ar23p_extended_syst_1022.df", keys2load, n_split, reprocess_df = False, reprocess_truth = True)

print("nudf has duplicates:", df['nudf'].index.duplicated().any())
print(df['nudf'].index.duplicated().sum(), "duplicate rows")

df['cc1pi'] = perform_truth_matching(df['cc1pi'], df['nudf'])
pot_weight_col = ('slc', 'wgt', '', '', '', '')
mc_tot_pot = df['hdr']['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = 4.560033e+18 / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
df['cc1pi'][pot_weight_col] = mc_pot_scale * np.ones(len(df['cc1pi']))

mc_evt_df = df['cc1pi']
mc_evt_df = mc_evt_df[build_event_cumulative_masks(mc_evt_df, sideband = "")["energy"]]

HelperFunctions.print_purity(mc_evt_df, ('truth','nu_categ','','','',''))


In [ ]:
print(mc_evt_df.slc.tmatch.idx)
print(mc_evt_df[mc_evt_df.truth.nu_categ == "cosmic"].slc.tmatch.idx)